# MS-VAR: Markov-Switching Vector Autoregression

Neste notebook, estendemos a abordagem de regime-switching para sistemas **multivariados**.
O modelo **MS-VAR** (Krolzig, 1997) permite que os parametros de um VAR mudem
de acordo com um regime latente governado por uma cadeia de Markov.

**Motivacao**: Em macroeconomia, multiplas variaveis (PIB, inflacao, taxa de juros)
frequentemente mudam de comportamento simultaneamente. O MS-VAR captura essa
co-movimentacao entre regimes.

**Conteudo:**
1. Motivacao para MS-VAR
2. MS(2)-VAR(1): estimacao
3. Regimes e impulso-resposta (IRF por regime)
4. Comparacao com VAR linear
5. Previsao condicional ao regime

**Referencias:**
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Hamilton, J.D. (1989). *A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle*. Econometrica.
- Ehrmann, M., Ellison, M. & Valla, N. (2003). Regime-dependent impulse response functions in a Markov-switching VAR model. *Economics Letters*, 78(3), 295-299.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('Bibliotecas carregadas com sucesso.')

## 1. Motivacao para MS-VAR

O modelo **VAR linear** assume que as relacoes entre variaveis sao **constantes** ao longo do tempo.
Porem, em muitas aplicacoes, essas relacoes mudam de acordo com o **estado da economia**:

- Em **expansao**: o multiplicador fiscal e menor, a curva de Phillips e mais plana
- Em **recessao**: choques se propagam de forma diferente, correlacoes mudam

O **MS-VAR** resolve isso permitindo que os parametros do VAR dependam de um regime $S_t$:

$$\mathbf{Y}_t = \boldsymbol{\mu}_{S_t} + \boldsymbol{\Phi}_{1,S_t} \mathbf{Y}_{t-1} + \cdots + \boldsymbol{\Phi}_{p,S_t} \mathbf{Y}_{t-p} + \boldsymbol{\epsilon}_t$$

onde $\boldsymbol{\epsilon}_t \sim N(\mathbf{0}, \boldsymbol{\Sigma}_{S_t})$ e $S_t$ segue uma cadeia de Markov.

### Taxonomia de Krolzig (1997)

| Sigla | O que muda entre regimes |
|-------|-------------------------|
| **MSI** | Intercepto $\boldsymbol{\mu}$ |
| **MSM** | Media $\boldsymbol{\mu}$ |
| **MSH** | Heteroscedasticidade $\boldsymbol{\Sigma}$ |
| **MSA** | Coeficientes autorregressivos $\boldsymbol{\Phi}$ |
| **MSIH** | Intercepto + Heteroscedasticidade |
| **MSMH** | Media + Heteroscedasticidade |

In [ ]:
# TODO: Carregue dados bivariados e visualize
# Dicas:
# - Gere dados MS-VAR sinteticos:
#     df = generate_ms_var(n=250, seed=57)
#     print(df.head())
#     print(f'\nRegimes: {df["true_regime"].value_counts().to_dict()}')
#
# - Plote as 2 series:
#     fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
#     for i, var in enumerate(['y1', 'y2']):
#         ax = axes[i]
#         ax.plot(df['date'], df[var], color='black', lw=0.8)
#         # Sombrear regimes
#         ax.fill_between(
#             df['date'], df[var].min(), df[var].max(),
#             where=df['true_regime'] == 2,
#             alpha=0.15, color='red', label='Regime 2'
#         )
#         ax.set_title(f'Serie {var}', fontsize=12)
#         ax.legend(loc='upper right')
#         ax.grid(True, alpha=0.3)
#     plt.tight_layout()
#     plt.show()

## 2. MS(2)-VAR(1)

Para um VAR bivariado com 2 regimes, o modelo e:

$$\begin{pmatrix} y_{1,t} \\ y_{2,t} \end{pmatrix} = \begin{pmatrix} \mu_{1,S_t} \\ \mu_{2,S_t} \end{pmatrix} + \begin{pmatrix} \phi_{11,S_t} & \phi_{12,S_t} \\ \phi_{21,S_t} & \phi_{22,S_t} \end{pmatrix} \begin{pmatrix} y_{1,t-1} \\ y_{2,t-1} \end{pmatrix} + \begin{pmatrix} \epsilon_{1,t} \\ \epsilon_{2,t} \end{pmatrix}$$

onde $\boldsymbol{\epsilon}_t \sim N(\mathbf{0}, \boldsymbol{\Sigma}_{S_t})$ e:

$$\boldsymbol{\Sigma}_{S_t} = \begin{pmatrix} \sigma^2_{11,S_t} & \sigma_{12,S_t} \\ \sigma_{12,S_t} & \sigma^2_{22,S_t} \end{pmatrix}$$

### Numero de parametros

Para MS(K)-VAR(p) com $n$ variaveis:
- Interceptos: $K \times n$
- Coeficientes AR: $K \times n^2 \times p$ (se switching) ou $n^2 \times p$ (se nao)
- Covariancias: $K \times \frac{n(n+1)}{2}$
- Transicao: $K(K-1)$

In [ ]:
# TODO: Estime MS(2)-VAR(1) com archbox
# Dicas:
# - Prepare os dados como array (T, n):
#     Y = df[['y1', 'y2']].values
#
# - Crie e ajuste o modelo:
#     model = MarkovSwitchingVAR(
#         endog=Y,
#         k_regimes=2,
#         order=1,
#         switching_mean=True,
#         switching_variance=True,
#     )
#     results = model.fit(method='em', maxiter=500, verbose=True)
#     print(results.summary())
#
# - Examine parametros por regime:
#     for regime_id, params in results.regime_params.items():
#         print(f'\n--- Regime {regime_id} ---')
#         for name, value in params.items():
#             print(f'  {name}: {value:.4f}')

## 3. Regimes e impulso-resposta

Uma das principais vantagens do MS-VAR e poder calcular **funcoes impulso-resposta (IRF)**
especificas para cada regime. Isso revela como choques se propagam de forma diferente
dependendo do estado da economia.

Para cada regime $s$, a IRF e calculada usando a matriz de coeficientes $\boldsymbol{\Phi}_{1,s}$
e a decomposicao de Cholesky de $\boldsymbol{\Sigma}_s$:

$$\text{IRF}_s(h) = \boldsymbol{\Phi}_{1,s}^h \cdot \mathbf{P}_s$$

onde $\mathbf{P}_s$ e o fator de Cholesky tal que $\boldsymbol{\Sigma}_s = \mathbf{P}_s \mathbf{P}_s'$.

**Interpretacao**: um choque unitario na variavel $j$ tem efeito diferente sobre a variavel $i$
dependendo de qual regime esta ativo — refletindo a **assimetria** das respostas economicas.

In [ ]:
# TODO: Calcule e plote IRF para cada regime
# Dicas:
# - Extraia parametros por regime:
#     regime_params = results.regime_params
#
# - Para cada regime, construa a matriz de coeficientes e calcule IRF:
#     horizonte = 12
#     var_names = ['y1', 'y2']
#
#     fig, axes = plt.subplots(2, 2, figsize=(12, 8))
#     fig.suptitle('Funcoes Impulso-Resposta por Regime', fontsize=14)
#
#     for regime_id, params in regime_params.items():
#         # Reconstruir a matriz Phi do regime
#         # (depende da estrutura de regime_params)
#         # Calcular IRF: irf[h] = Phi^h @ cholesky(Sigma)
#         pass  # Implementar calculo de IRF
#
#     # Plote IRF: resposta de y_i a choque em y_j, para cada regime
#     # Compare os regimes lado a lado
#     plt.tight_layout()
#     plt.show()

## 4. Comparacao com VAR linear

Para avaliar se a mudanca de regime e empiricamente relevante, comparamos o MS-VAR
com um **VAR linear simples** (sem regimes):

$$\mathbf{Y}_t = \boldsymbol{\mu} + \boldsymbol{\Phi}_1 \mathbf{Y}_{t-1} + \boldsymbol{\epsilon}_t$$

**Criterios de comparacao:**
- **AIC/BIC**: penalizam complexidade adicional do MS-VAR
- **Log-verossimilhanca**: MS-VAR sempre tera $\ell$ maior, mas nao necessariamente melhor fit ajustado
- **Qualidade da previsao**: comparar RMSE fora da amostra

O VAR linear e um caso especial do MS-VAR quando $K=1$ (um unico regime).

In [ ]:
# TODO: Compare MS-VAR com VAR linear simples
# Dicas:
# - Estime VAR linear (MS-VAR com k_regimes=1 ou OLS direto):
#     model_linear = MarkovSwitchingVAR(
#         endog=Y,
#         k_regimes=1,
#         order=1,
#     )
#     results_linear = model_linear.fit(method='em', maxiter=500)
#
# - Compare criterios de informacao:
#     print('Comparacao VAR linear vs MS(2)-VAR(1):')
#     print(f'  VAR linear: AIC={results_linear.aic:.2f}, BIC={results_linear.bic:.2f}')
#     print(f'  MS(2)-VAR:  AIC={results.aic:.2f}, BIC={results.bic:.2f}')
#     print(f'  Log-lik VAR: {results_linear.loglike:.2f}')
#     print(f'  Log-lik MS:  {results.loglike:.2f}')
#
# - Plote probabilidades suavizadas do MS-VAR:
#     fig = plot_regime_probabilities(
#         dates=df['date'].values,
#         series=df['y1'].values,
#         probabilities=results.smoothed_probs,
#         regime_labels=['Regime 1', 'Regime 2'],
#         title='MS(2)-VAR(1): Probabilidades de Regime',
#     )
#     plt.show()

## 5. Previsao condicional ao regime

O MS-VAR permite fazer previsoes **condicionais** ao regime ativo:

$$E[\mathbf{Y}_{T+h} \mid S_{T+1} = s, \mathcal{Y}_T] = \boldsymbol{\mu}_s + \boldsymbol{\Phi}_{1,s} \mathbf{Y}_T$$

Isso e util para **analise de cenarios**:
- "Se a economia permanecer em expansao, qual e a previsao?"
- "Se entrar em recessao, como as variaveis se comportam?"

A previsao **incondicional** (ponderada por regime) e:

$$E[\mathbf{Y}_{T+h} \mid \mathcal{Y}_T] = \sum_{s=1}^{K} P(S_{T+h} = s \mid \mathcal{Y}_T) \cdot E[\mathbf{Y}_{T+h} \mid S_{T+h} = s, \mathcal{Y}_T]$$

In [ ]:
# TODO: Faca previsao condicional a cada regime
# Dicas:
# - Horizonte de previsao:
#     h = 8
#     Y_last = Y[-1]
#
# - Previsao condicional a cada regime:
#     forecasts = {}
#     for regime_id, params in results.regime_params.items():
#         # Extrair mu e Phi do regime
#         # Calcular previsoes iterativas: Y_{T+h} = mu + Phi @ Y_{T+h-1}
#         pass
#
# - Plote previsoes para cada regime:
#     fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#     for i, var in enumerate(['y1', 'y2']):
#         ax = axes[i]
#         ax.plot(range(h), forecasts[1][:, i], 'b-o', label='Regime 1')
#         ax.plot(range(h), forecasts[2][:, i], 'r-o', label='Regime 2')
#         ax.set_title(f'Previsao condicional: {var}')
#         ax.set_xlabel('Horizonte')
#         ax.legend()
#         ax.grid(True, alpha=0.3)
#     plt.tight_layout()
#     plt.show()

## Conclusao

Neste notebook, aprendemos:

- A **motivacao** para modelos MS-VAR: capturar mudancas estruturais em sistemas multivariados
- Como estimar um **MS(2)-VAR(1)** com o algoritmo EM
- Como calcular **IRFs por regime**, revelando assimetrias nas respostas a choques
- Como comparar com um **VAR linear** usando AIC/BIC
- Como fazer **previsoes condicionais** a cada regime (analise de cenarios)

No proximo notebook, combinamos regime-switching com **volatilidade condicional** no modelo **MS-GARCH**.

### Referencias

- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, 57(2), 357-384.
- Ehrmann, M., Ellison, M. & Valla, N. (2003). Regime-dependent impulse response functions in a Markov-switching VAR model. *Economics Letters*, 78(3), 295-299.
- Haas, M., Mittnik, S., & Paolella, M.S. (2004). A New Approach to Markov-Switching GARCH Models. *Journal of Financial Econometrics*, 2(4), 493-530.